## 최종 선정된 입지를 기반으로 차량 루트 최적화하기

In [20]:
import numpy as np
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
from math import radians, sin, cos, sqrt, atan2, ceil
import networkx as nx
import osmnx as ox

In [21]:
candidate_df = pd.read_csv('data/final_candidates_v1.csv', encoding='cp949')
demand_df = pd.read_csv('data/demand_v1.csv', encoding='utf-8')
# Longitude, Latitude 컬럼 이름 바꾸기
demand_df.rename(columns={'Longitude': '경도', 'Latitude': '위도'}, inplace=True)
assign_df = pd.read_csv('data/assignment_v5.csv', encoding='cp949')

In [22]:
demand_df.head()

,k-아파트명,경도,위도,수요
0,우리유앤미,126.959639,37.500668,1
1,송파파인타운13단지,127.129179,37.476897,0
2,오금현대백조(임대),127.128775,37.508906,0
3,개봉건영,126.840675,37.501162,1
4,월계동원베네스트,127.058220,37.631732,0


### 1) 거리행렬함수

In [23]:
# def build_osm_vrp_distance_matrix(G, locations, weight="length"):
#     """
#     locations: [(lat, lon), ...]
#     첫 번째는 depot(MFC), 나머지는 수요지
#     weight:
#       - "length": 도로거리(m)
#       - "travel_time_min": 도로시간(분)
#     반환:
#       - matrix_km if length 기준
#       - matrix_min if travel_time_min 기준
#     """

#     # 좌표를 OSM nearest node로 매핑
#     nodes = ox.distance.nearest_nodes(
#         G,
#         X=[lon for lat, lon in locations],
#         Y=[lat for lat, lon in locations]
#     )

#     n = len(nodes)
#     matrix = np.full((n, n), np.inf)

#     for i, source in enumerate(nodes):
#         lengths = nx.single_source_dijkstra_path_length(
#             G,
#             source=source,
#             weight=weight
#         )

#         for j, target in enumerate(nodes):
#             matrix[i, j] = lengths.get(target, np.inf)

#     # length는 m 단위라 km 변환
#     if weight == "length":
#         matrix = matrix / 1000

#     return matrix

In [24]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = (
        sin(dlat / 2) ** 2
        + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    )
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c


def build_distance_matrix_km(locations):
    """
    locations: [(lat, lon), ...]
    """
    n = len(locations)
    matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i != j:
                matrix[i, j] = haversine_km(
                    locations[i][0], locations[i][1],
                    locations[j][0], locations[j][1]
                )

    return matrix

### 2) OR-Tools CVRP 함수

In [25]:
def solve_cvrp_ortools(
    distance_matrix_km,
    demands,
    vehicle_capacity=24,
    num_vehicles=None,
    depot_index=0,
    time_limit_sec=30
):
    """
    CVRP:
    - depot_index: MFC 위치
    - demands[0]은 depot 수요이므로 0이어야 함
    """

    n = len(distance_matrix_km)

    if num_vehicles is None:
        total_demand = sum(demands)
        num_vehicles = max(1, ceil(total_demand / vehicle_capacity))

    # OR-Tools는 정수 cost를 선호하므로 km → m 변환
    distance_matrix_m = (distance_matrix_km * 1000).astype(int)

    manager = pywrapcp.RoutingIndexManager(
        n,
        num_vehicles,
        depot_index
    )

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return int(distance_matrix_m[from_node][to_node])

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        from_node = manager.IndexToNode(from_index)
        return int(demands[from_node])

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        [vehicle_capacity] * num_vehicles,
        True,
        "Capacity"
    )

    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_params.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_params.time_limit.seconds = time_limit_sec

    solution = routing.SolveWithParameters(search_params)

    if solution is None:
        return None

    routes = []

    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)

        route_nodes = []
        route_distance_m = 0
        route_load = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route_nodes.append(node)
            route_load += demands[node]

            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance_m += routing.GetArcCostForVehicle(
                previous_index, index, vehicle_id
            )

        route_nodes.append(manager.IndexToNode(index))

        # 실제 방문 수요지가 없는 차량은 제외
        if len(route_nodes) > 2:
            routes.append({
                "vehicle_id": vehicle_id,
                "route_nodes": route_nodes,
                "route_distance_km": route_distance_m / 1000,
                "route_load": route_load
            })

    return routes

### 3) MFC별 VRP 실행 함수

In [26]:
def run_vrp_by_mfc(
    assign_df,
    candidate_df,
    demand_df,
    vehicle_capacity=24, # 1톤 트럭의 capacity: 24가구
    demand_col="수요",
    candidate_name_col="학교명",
    lat_col="위도",
    lon_col="경도",
    time_limit_sec=30
):
    assign_df = assign_df.copy()

    all_route_rows = []
    summary_rows = []

    for candidate_index, group in assign_df.groupby("candidate_index"):
        school_name = candidate_df.iloc[candidate_index][candidate_name_col]

        # depot = MFC
        depot_lat = candidate_df.iloc[candidate_index][lat_col]
        depot_lon = candidate_df.iloc[candidate_index][lon_col]

        demand_indices = group["demand_index"].tolist()
        demand_sub = demand_df.iloc[demand_indices].copy()

        # locations: 첫 번째는 depot
        locations = [(depot_lat, depot_lon)] + list(
            zip(demand_sub[lat_col], demand_sub[lon_col])
        )

        # demands: depot은 0
        # OR-Tools capacity는 정수 수요를 요구하므로 반올림/올림 필요
        demands = [0] + np.ceil(demand_sub[demand_col].to_numpy()).astype(int).tolist()

        total_demand = sum(demands)
        num_vehicles = max(1, ceil(total_demand / vehicle_capacity))

        distance_matrix_km = build_distance_matrix_km(locations)

        routes = solve_cvrp_ortools(
            distance_matrix_km=distance_matrix_km,
            demands=demands,
            vehicle_capacity=vehicle_capacity,
            num_vehicles=num_vehicles,
            depot_index=0,
            time_limit_sec=time_limit_sec
        )

        if routes is None:
            summary_rows.append({
                "candidate_index": candidate_index,
                "candidate_name": school_name,
                "status": "failed",
                "num_demand_nodes": len(demand_sub),
                "total_demand": total_demand,
                "num_vehicles": num_vehicles,
                "total_route_distance_km": np.nan,
                "max_route_distance_km": np.nan,
            })
            continue

        total_route_distance = sum(r["route_distance_km"] for r in routes)
        max_route_distance = max(r["route_distance_km"] for r in routes) if routes else 0

        summary_rows.append({
            "candidate_index": candidate_index,
            "candidate_name": school_name,
            "status": "success",
            "num_demand_nodes": len(demand_sub),
            "total_demand": total_demand,
            "num_vehicles": len(routes),
            "total_route_distance_km": total_route_distance,
            "max_route_distance_km": max_route_distance,
        })

        for r in routes:
            route_nodes = r["route_nodes"]

            # route_nodes에서 0은 depot, 나머지는 demand_sub의 위치 index
            visited_demand_indices = []
            for node in route_nodes:
                if node == 0:
                    continue
                original_demand_index = demand_indices[node - 1]
                visited_demand_indices.append(original_demand_index)

            all_route_rows.append({
                "candidate_index": candidate_index,
                "candidate_name": school_name,
                "vehicle_id": r["vehicle_id"],
                "route_nodes": route_nodes,
                "visited_demand_indices": visited_demand_indices,
                "route_load": r["route_load"],
                "route_distance_km": r["route_distance_km"],
            })

    vrp_summary_df = pd.DataFrame(summary_rows)
    vrp_routes_df = pd.DataFrame(all_route_rows)

    return vrp_summary_df, vrp_routes_df

### 4) 실행

In [27]:
vrp_summary_df, vrp_routes_df = run_vrp_by_mfc(
    assign_df=assign_df,
    candidate_df=candidate_df,
    demand_df=demand_df,
    vehicle_capacity=24,
    demand_col="수요",
    candidate_name_col="학교명",
    lat_col="위도",
    lon_col="경도",
    time_limit_sec=30
)

print(vrp_summary_df.head())
print(vrp_routes_df.head())

   candidate_index     candidate_name   status  num_demand_nodes  \
0                0  상명대학교사범대학부속여자고등학교  success                32   
1                1            대원국제중학교  success                47   
2                2          서울디지텍고등학교  success               159   
3               10            보성여자중학교  success               147   
4               15           영신여자고등학교  success               106   

   total_demand  num_vehicles  total_route_distance_km  max_route_distance_km  
0            45             2                   15.647                  7.864  
1           106             5                  126.912                 29.653  
2           257            11                  121.524                 20.804  
3           239            10                  262.008                 32.646  
4           252            11                  578.789                 56.693  
   candidate_index     candidate_name  vehicle_id  \
0                0  상명대학교사범대학부속여자고등학교           0   
1    

In [29]:
import folium

def visualize_route(candidate_df, demand_df, route_info):
    candidate_index = route_info["candidate_index"]

    depot_row = candidate_df.iloc[candidate_index]
    depot_lat = depot_row["위도"]
    depot_lon = depot_row["경도"]
    depot_name = depot_row["학교명"]

    # visited_demand_indices는 이미 방문 순서대로 된 원본 demand index
    visited_demand_indices = route_info["visited_demand_indices"]

    m = folium.Map(
        location=[depot_lat, depot_lon],
        zoom_start=13
    )

    # MFC 표시
    folium.Marker(
        location=[depot_lat, depot_lon],
        popup=f"MFC: {depot_name}",
        icon=folium.Icon(color="red", icon="home")
    ).add_to(m)

    # 경로 좌표: depot에서 시작
    route_coords = [(depot_lat, depot_lon)]

    for order, demand_idx in enumerate(visited_demand_indices, start=1):
        demand_row = demand_df.iloc[demand_idx]

        lat = demand_row["위도"]
        lon = demand_row["경도"]

        route_coords.append((lat, lon))

        folium.Marker(
            location=[lat, lon],
            popup=f"{order}번째 방문 / demand_index={demand_idx}",
            icon=folium.Icon(color="blue", icon="info-sign")
        ).add_to(m)

    # depot으로 복귀
    route_coords.append((depot_lat, depot_lon))

    folium.PolyLine(
        locations=route_coords,
        weight=4,
        opacity=0.8
    ).add_to(m)

    return m

In [33]:
# 예시: 첫 번째 경로 시각화(상명대학교사범대학부속여자고등학교)
if not vrp_routes_df.empty:
    example_route = vrp_routes_df.iloc[0]
    route_map = visualize_route(candidate_df, demand_df, example_route)
    route_map.save("example_route.html")

In [34]:
route_map